# rahulk-flow-video — Kaggle T4 training

Flow-matching video model. This notebook clones the repo, attaches the cached
Moving MNIST dataset (no raw download), reads the HF token from Kaggle Secrets,
trains, and pushes checkpoints to HF Hub each save.

## Before you Run All — check the sidebar (⋮ / Settings):
1. **Accelerator = GPU T4** (Session options ▸ Accelerator).
2. **Internet = On** (needed to clone the repo + push to HF).
3. **Add data** ▸ search `rahulkhunte/moving-mnist-seqfirst` ▸ Add.
4. **Add-ons ▸ Secrets** ▸ add a secret named `HF_TOKEN` = your Hugging Face
   write token (https://huggingface.co/settings/tokens). Never paste it in a cell.

**Run the smoke test first** (`SMOKE = True`): 200 steps that confirm GPU + data +
HF push + HF pull/resume all work end-to-end. Only then flip `SMOKE = False` for
the real 30–50k-step run.

## 0 · Config toggle

In [ ]:
# Flip to False for the real 30-50k run.
SMOKE = True

REPO_URL = 'https://github.com/rahulkhunte/rahulk-flow-video.git'
REPO_DIR = '/kaggle/working/rahulk-flow-video'
DATA_NPY = '/kaggle/input/moving-mnist-seqfirst/moving_mnist_seqfirst.npy'
HF_REPO  = 'rahulkhunte/rahulk-flow-video-ckpts'   # private; auto-created on first push
CKPT_DIR = '/kaggle/working/checkpoints'

## 1 · Clone the repo

In [ ]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

## 2 · Install deps
Torch / numpy / pyyaml ship with the Kaggle image; we only add `huggingface_hub`.

In [ ]:
!pip install -q huggingface_hub

## 3 · Attach the dataset (no raw download)
The dataset is mounted read-only under `/kaggle/input`. We auto-discover the
`.npy` (the exact mount path can vary), point the config's `data_cache` at it,
and `MovingMNIST` mmaps it — nothing is copied or regenerated.

In [ ]:
import glob, numpy as np
print('/kaggle/input:', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'MISSING')
hits = glob.glob('/kaggle/input/**/moving_mnist_seqfirst.npy', recursive=True)
assert hits, ('dataset npy not found under /kaggle/input — add data '
              'rahulkhunte/moving-mnist-seqfirst in the sidebar (Session ▸ Add data).')
DATA_NPY = hits[0]                       # override the cell-0 default with the real path
shp = np.load(DATA_NPY, mmap_mode='r').shape
print('cache:', DATA_NPY, '->', shp)
assert shp[1:] == (20, 64, 64), f'unexpected shape {shp} — expected (N, 20, 64, 64) sequence-first'

## 4 · HF token from Kaggle Secrets
Read the `HF_TOKEN` secret, expose it as an env var (`train.py` reads
`os.environ['HF_TOKEN']`), and derive the HF repo from the token's OWN
namespace via `whoami` — so it works whatever your HF username is, and never
assumes it matches your Kaggle name. The token is never printed.

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
from huggingface_hub import HfApi
who = HfApi(token=os.environ['HF_TOKEN']).whoami()
HF_USER = who['name']
HF_REPO = f'{HF_USER}/rahulk-flow-video-ckpts'   # override cell-0 default with real namespace
role = who.get('auth', {}).get('accessToken', {}).get('role', '?')
print('HF user:', HF_USER, '| token role:', role, '| repo:', HF_REPO)
assert role in ('write', 'admin', 'fineGrained', '?'), f'HF token role {role!r} may be read-only — need a WRITE token.'

## 5 · GPU check
Must be a **T4** (or newer). Kaggle's PyTorch dropped Pascal support, so a
P100 (`sm_60`) crashes at the first kernel launch — we fail fast here instead.

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU is OFF — set Accelerator = GPU T4 x2 in Session options.'
name = torch.cuda.get_device_name(0); cap = torch.cuda.get_device_capability(0)
print('GPU:', name, '| compute', cap)
assert cap >= (7, 0), (f'{name} is compute {cap} — Kaggle torch needs sm_70+. '
                       'Set Accelerator = GPU T4 x2 in Session options (P100 is unsupported).')

## 6 · Build the run config
Start from the repo `config.yaml`, then override data/checkpoint/HF paths. In
SMOKE mode we also shrink the run and save often so the HF push + resume round-
trip actually gets exercised inside 200 steps.

In [ ]:
import yaml
cfg = yaml.safe_load(open('config.yaml'))
cfg['data_cache']    = DATA_NPY
cfg['data_src']      = ''            # cache present; src never touched
cfg['checkpoint_dir'] = CKPT_DIR
cfg['hf_repo']       = HF_REPO

if SMOKE:
    cfg['max_steps']     = 200
    cfg['save_every']    = 100        # saves at step 100 and 200 -> HF push happens
    cfg['hf_push_every'] = 100        # also lay down a permanent milestone
    cfg['log_every']     = 20

yaml.safe_dump(cfg, open('run_config.yaml', 'w'))
print(f"SMOKE={SMOKE}  max_steps={cfg['max_steps']}  save_every={cfg['save_every']}  "
      f"batch_size={cfg['batch_size']}  hf_repo={cfg['hf_repo']}")

## 7 · Train
On startup `train()` pulls `resume_latest.pth` from HF if it exists, so re-
running this cell after a dropped session continues where it left off. Each save
pushes the latest checkpoint (+ a milestone every `hf_push_every` steps).

In [ ]:
import importlib, train as train_mod
importlib.reload(train_mod)
losses = train_mod.train('run_config.yaml')
print(f'\ntrain done — {len(losses)} steps, last loss {losses[-1]:.4f}')

## 8 · SMOKE only · HF pull + resume round-trip
Wipe the local checkpoints so the next launch has nothing on disk, then train a
few more steps. It MUST pull `resume_latest.pth` (step 200) back from HF and
continue — proving a fresh Kaggle session self-heals. Watch for
`Pulled latest checkpoint from HF` and `Resumed at step 200`.

In [ ]:
if SMOKE:
    import shutil
    shutil.rmtree(CKPT_DIR, ignore_errors=True)          # simulate a fresh session
    cfg['max_steps'] = 260
    yaml.safe_dump(cfg, open('run_config.yaml', 'w'))
    importlib.reload(train_mod)
    losses = train_mod.train('run_config.yaml')          # pulls step 200 from HF, -> 260
    print(f'\nresume round-trip OK — ended at {len(losses)} loss entries')
else:
    print('skipped (real run)')

## 9 · Eyeball progress — sample from the latest EMA
Pull `ema_latest.pth` from HF and render sample GIFs to watch the digits
sharpen. Run it any time — including in a **separate free CPU session** while
the batch T4 job trains: just run cell 1 (clone) + cell 2 (pip), then this
cell (skip the data/GPU/train cells). Forced to CPU — sampling is light, needs
no GPU quota, and sidesteps the P100 torch break. Early on it's near-noise;
coherent motion emerges over the run.

In [ ]:
# self-contained: (re)establish HF token + repo if this is a fresh session
import os, glob, importlib
if not os.environ.get('HF_TOKEN'):
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
try:
    HF_REPO
except NameError:
    from huggingface_hub import HfApi
    HF_REPO = f"{HfApi(token=os.environ['HF_TOKEN']).whoami()['name']}/rahulk-flow-video-ckpts"

from huggingface_hub import hf_hub_download
import sample as sample_mod; importlib.reload(sample_mod)

cfg_path = 'run_config.yaml' if os.path.exists('run_config.yaml') else 'config.yaml'
ema_path = hf_hub_download(HF_REPO, 'ema_latest.pth', repo_type='model',
                           token=os.environ['HF_TOKEN'], local_dir='/kaggle/working/pulled')
sample_mod.sample(cfg_path, ckpt=ema_path, n=4,
                  out_dir='/kaggle/working/samples', device='cpu')

from IPython.display import Image as IPyImage, display
for g in sorted(glob.glob('/kaggle/working/samples/sample_*_*.gif')):
    print(g); display(IPyImage(filename=g))

## 10 · Real run
When the smoke test passes: set `SMOKE = False` at the top, **Run All**. The
config's `max_steps` (50k) and `save_every` (5k) drive a full run, checkpoints
streaming to HF. If the T4's 12h window cuts it off, just Run All again — it
resumes from HF automatically. Milestone: coherent moving digits without flicker.